In [3]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.medgan.models import MEDGAN
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset


ROOT = Path.cwd().parent

print("Project root:", ROOT)


raw_file =  ROOT / "katabatic" / "datasets" / "car.csv"

processed_file = (
    ROOT
    / "preprocessed_data"
    / "car_medgan_paper_params.csv"
)

artifact_dir = ROOT / "artifacts"

processed_file.parent.mkdir(
    parents=True,
    exist_ok=True,
)


preprocess_dataset(
    str(raw_file),
    str(processed_file),
)


store = LocalArtifactStore(
    str(artifact_dir)
)


model = MEDGAN(
    ae_pretrain_epochs=100,
    gan_epochs=1000,
    batch_size=1000,
    ae_lr=1e-3,
    generator_lr=1e-3,
    discriminator_lr=1e-3,
    latent_dim=128,
    generator_hidden_dim=128,
    generator_num_layers=2,
    discriminator_num_layers=2,
    bn_decay=0.99,
    dropout=0.0,
    random_state=42,
)


pipeline = TrainTestSplitPipeline(
    model=model
)

pipeline._evaluations = []


results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="car_medgan_paper_params",
    artifact_store=store,
    model_name="medgan",
)


print(results)

Project root: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\car.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\preprocessed_data\car_medgan_paper_params.csv
Loaded data with shape: (1728, 7)


INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Training MedGAN Model
INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Loaded training data: (1382, 6)
INFO:katabatic.models.medgan.models:Categorical columns: ['0', '1', '2', '3', '4', '5', '6']
INFO:katabatic.models.medgan.models:Continuous columns: []
INFO:katabatic.models.medgan.models:Data normalized to [0, 1] range
INFO:katabatic.models.medgan.models:Original range: [0.00, 3.00]
INFO:katabatic.models.medgan.models:Normalized range: [0.00, 1.00]
INFO:katabatic.models.medgan.models:
Phase 1: Pretraining Autoencoder for 100 epochs...


Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved dataset artifact under datasets/car_medgan_paper_params/split-20260809-080141


INFO:katabatic.models.medgan.models:Epoch 1/100: AE Loss = 0.694203
INFO:katabatic.models.medgan.models:Epoch 10/100: AE Loss = 0.627351
INFO:katabatic.models.medgan.models:Epoch 20/100: AE Loss = 0.572984
INFO:katabatic.models.medgan.models:Epoch 30/100: AE Loss = 0.525361
INFO:katabatic.models.medgan.models:Epoch 40/100: AE Loss = 0.483533
INFO:katabatic.models.medgan.models:Epoch 50/100: AE Loss = 0.447840
INFO:katabatic.models.medgan.models:Epoch 60/100: AE Loss = 0.416997
INFO:katabatic.models.medgan.models:Epoch 70/100: AE Loss = 0.394295
INFO:katabatic.models.medgan.models:Epoch 80/100: AE Loss = 0.377045
INFO:katabatic.models.medgan.models:Epoch 90/100: AE Loss = 0.365303
INFO:katabatic.models.medgan.models:Epoch 100/100: AE Loss = 0.356097
INFO:katabatic.models.medgan.models:
Phase 2: Training GAN for 1000 epochs...
INFO:katabatic.models.medgan.models:Epoch 1/1000: D Loss = 1.376434, G Loss = 0.700370
INFO:katabatic.models.medgan.models:Epoch 100/1000: D Loss = 0.208309, G Los

{'message': 'Train test split pipeline executed successfully.', 'dataset_ref': DatasetRef(dataset_name='car_medgan_paper_params', dataset_version='split-20260809-080141'), 'model_ref': ModelRef(model_name='medgan', dataset_name='car_medgan_paper_params', dataset_version='split-20260809-080141', train_run_id='train-20260809-080141'), 'evaluation_refs': []}


In [8]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline


split_dir = (
    ROOT
    / "artifacts"
    / "datasets"
    / "car_medgan_paper_params"
    / "split-20260809-080141"
)


x_train = pd.read_csv(
    split_dir / "train" / "x_train.csv"
)

y_train = pd.read_csv(
    split_dir / "train" / "y_train.csv"
)

real_data = pd.concat(
    [x_train, y_train],
    axis=1,
)


x_test = pd.read_csv(
    split_dir / "test" / "x_test.csv"
)

y_test = pd.read_csv(
    split_dir / "test" / "y_test.csv"
)

test_data = pd.concat(
    [x_test, y_test],
    axis=1,
)


synthetic_data = model.sample(
    len(real_data),
    seed=42,
)


evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=[
        "0",
        "1",
        "2",
        "3",
        "4",
        "5",
    ],
    continuous_cols=[],
)


evaluation_results = evaluation_pipeline.run(
    real_data=real_data,
    synthetic_data=synthetic_data,
    target_col="6",
    test_data=test_data,
    model=model,
)


print(evaluation_results)


Running fidelity evaluation...

=== Fidelity Evaluation ===
Overall fidelity score: 0.8464

Categorical JSD (lower = better)  ->  score: 0.8464
  0                              JSD = 0.0722
  1                              JSD = 0.1727
  2                              JSD = 0.1579
  3                              JSD = 0.2003
  4                              JSD = 0.1967
  5                              JSD = 0.1220
  avg                            JSD = 0.1536

Running utility evaluation...


c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\O


=== Utility Evaluation ===
Overall utility score: 0.7024

Classifier   Metric     TSTR mean    TRTR mean    Delta   
--------------------------------------------------------
LR           accuracy   0.6329       0.6884       0.0555
LR           f1         0.5429       0.6097       0.0668
DT           accuracy   0.4642       0.9734       0.5092
DT           f1         0.4851       0.9730       0.4879
RF           accuracy   0.5434       0.9671       0.4237
RF           f1         0.5201       0.9668       0.4467
LinearSVM    accuracy   0.6595       0.7023       0.0428
LinearSVM    f1         0.5559       0.6239       0.068
MLP          accuracy   0.5486       0.9717       0.4231
MLP          f1         0.5195       0.9717       0.4522

Running diversity evaluation...

=== Diversity Evaluation ===
Overall diversity score: 0.9773

Category Coverage (% of real categories in synth)
  0                              100.0%
  1                              100.0%
  2                           